# CT Image Denoising with Deep Learning

In this notebook, we train a convolutional neural network to remove noise from CT images.

The neural network receives a noisy image as input and tries to reconstruct the corresponding clean image.

The workflow is:

1. Load the experiment configuration
2. Load the training, validation, and test datasets
3. Create a neural network
4. Choose a loss function and optimizer
5. Train the network
6. Monitor training and validation loss
7. Visualize denoising results
8. Evaluate the final model on the test set

In [ ]:
from pathlib import Path
import os
import sys


# Find the project root
current_dir = Path.cwd()

if (current_dir / "train.py").exists():
    # Notebook is already running from the project root
    project_root = current_dir

elif (current_dir.parent / "train.py").exists():
    # Notebook is running from experiments/
    project_root = current_dir.parent

else:
    raise RuntimeError("Could not find the project root.")


# Make the project root the working directory
os.chdir(project_root)

# Make project modules importable
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


print("Project root:", project_root)
print("Working directory:", Path.cwd())


from pathlib import Path

import torch
from utils.dataloader import create_dataloaders
from utils.misc import load_config, set_seed
from utils.model_configs import get_loss, get_optimizer, get_scheduler
from utils.plot_utils import plot_training_history, save_denoising_examples, plot_psnr_history, plot_noisy_clean_examples, plot_augmentation_examples, plot_uncertainty_examples
from utils.print_utils import print_config_summary
from train import (
    evaluate,
    train_model,
    evaluate_variational,
    train_model_variational,
    predict_with_uncertainty,
)
from models import create_model
from utils.experiment import create_experiment, save_history
from utils.metrics import calculate_psnr

## Configuration

The settings for the experiment are stored in `config.yaml`.

This allows us to change properties such as:

- model architecture
- batch size
- number of epochs
- optimizer
- learning rate
- loss function

without modifying the training code itself.

In [ ]:
config = load_config("config.yaml")

# Seed Python, NumPy and PyTorch so a run can be repeated
set_seed(config["seed"])

print_config_summary(config)

experiment_name = "00_baseline"

run_dir = create_experiment(
    config=config,
    experiment_name=experiment_name,
)

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(config)


# ============================================================
# Dataset sizes
# ============================================================

print("Dataset sizes")
print("-------------")
print("Training samples:  ", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:      ", len(test_loader.dataset))

print()


# ============================================================
# Number of batches
# ============================================================

print("Number of batches")
print("-----------------")
print("Training batches:  ", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:      ", len(test_loader))

print()


# ============================================================
# Noise levels
# ============================================================

print("Noise levels")
print("------------")
print(
    "Train/validation σ:",
    config["noise"]["train_val_std"],
)
print(
    "Test σ:            ",
    config["noise"]["test_std"],
)

print()


# ============================================================
# Inspect one training batch
# ============================================================

noisy_images, clean_images = next(
    iter(train_loader)
)

print("Training batch")
print("--------------")
print("Noisy batch shape:", noisy_images.shape)
print("Clean batch shape:", clean_images.shape)

print()

print(
    "Noisy min/max:",
    noisy_images.min().item(),
    noisy_images.max().item(),
)

print(
    "Clean min/max:",
    clean_images.min().item(),
    clean_images.max().item(),
)

In [ ]:
noisy_images, clean_images = next(
    iter(train_loader)
)

plot_noisy_clean_examples(
    noisy_images=noisy_images,
    clean_images=clean_images,
    noise_std=config["noise"]["train_val_std"],
    num_images=4,
)

In [ ]:
if config["augmentation"]["enabled"]:

    plot_augmentation_examples(
        dataset=train_loader.dataset,
        index=450,
        num_examples=4,
    )

else:

    print(
        "Data augmentation is disabled. "
        "Set augmentation.enabled=true in config.yaml "
        "to enable it."
    )

In [ ]:
# ============================================================
# Training setup
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Variational models return (prediction, kl) and use the KL-aware training loop
is_variational = config["model"]["name"].startswith("variational")

print(f"Loading Model: {config['model']['name']}")
model = create_model(config["model"])
model = model.to(device)
# print(model)



criterion = get_loss(
    config["training"]["loss"]
)

optimizer = get_optimizer(
    optimizer_name=config["training"]["optimizer"],
    model=model,
    learning_rate=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
    rho_learning_rate=config["training"].get("rho_learning_rate"),
)

scheduler = get_scheduler(
    scheduler_name=config["training"].get("scheduler", "none"),
    optimizer=optimizer,
    epochs=config["training"]["epochs"],
)


num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

num_trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Total parameters:     {num_parameters:,}")
print(f"Trainable parameters: {num_trainable_parameters:,}")

In [ ]:
if is_variational:

    history = train_model_variational(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epochs=config["training"]["epochs"],
        kl_weight=config["training"]["kl_weight"],
        output_dir=run_dir,
        save_best_model=config["output"]["save_best_model"],
        scheduler=scheduler
    )

else:

    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epochs=config["training"]["epochs"],
        output_dir=run_dir,
        save_best_model=config["output"]["save_best_model"],
        scheduler=scheduler
    )

In [ ]:
plot_training_history(history, save_path=run_dir / "loss_curve.png")

plot_psnr_history(history, save_path=run_dir / "psnr_curve.png")

In [ ]:
checkpoint_path = (run_dir / "best_model.pt")

model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

print("Loaded:", checkpoint_path)

In [ ]:
if config["output"]["save_example_images"]:

    save_denoising_examples(
        model=model,
        dataloader=val_loader,
        device=device,
        save_path=run_dir / "val_denoising_examples.png",
        num_images=4,
    )

In [ ]:
if is_variational:

    # Posterior-mean network (sample=False); reconstruction loss only
    test_metrics = evaluate_variational(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device,
        dataset_size=len(train_loader.dataset),
        kl_weight=config["training"]["kl_weight"],
        sample=False,
        description="Test",
    )

    test_loss = test_metrics["reconstruction"]
    test_psnr = test_metrics["psnr"]

else:

    test_loss, test_psnr = evaluate(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device,
        description="Test",
    )

print(f"Test loss: {test_loss:.6f}")
print(f"Test PSNR: {test_psnr:.2f} dB")

In [ ]:
save_denoising_examples(
    model=model,
    dataloader=test_loader,
    device=device,
    save_path=run_dir / "test_examples.png",
    num_images=4,
)

## Uncertainty (variational models only)

A variational model has a distribution over its weights. Repeating the forward pass with freshly sampled weights gives slightly different predictions.

- **Predictive mean**: the average over the sampled predictions
- **Epistemic std**: the spread between the sampled predictions

The std only measures disagreement between weight samples. It is not calibrated, and high or low std is not proof that the uncertainty is right. The correlation below shows whether the std is related to the actual error at all.

In [ ]:
if is_variational:

    number_of_samples = 30

    noisy_images, clean_images = next(iter(val_loader))
    noisy_images = noisy_images.to(device)
    clean_images = clean_images.to(device)

    predictive_mean, epistemic_variance = predict_with_uncertainty(
        model,
        noisy_images,
        number_of_samples=number_of_samples,
    )

    plot_uncertainty_examples(
        noisy_images=noisy_images,
        clean_images=clean_images,
        predictive_mean=predictive_mean,
        epistemic_variance=epistemic_variance,
        save_path=run_dir / "val_uncertainty.png",
        num_images=4,
    )

    error = (predictive_mean - clean_images).abs()
    epistemic_std = epistemic_variance.sqrt()

    correlation = torch.corrcoef(
        torch.stack([epistemic_std.flatten(), error.flatten()])
    )[0, 1].item()

    print(f"Samples per image:        {number_of_samples}")
    print(f"Mean epistemic std:       {epistemic_std.mean().item():.5f}")
    print(f"Mean absolute error:      {error.mean().item():.5f}")
    print(f"PSNR of predictive mean:  {calculate_psnr(predictive_mean, clean_images):.2f} dB")
    print(f"Pixel-wise corr(std, |error|): {correlation:.3f}")

else:

    print("Uncertainty is only available for variational models.")

### Behaviour at different noise levels

The same clean validation images are corrupted with Gaussian noise of different strengths. The model was trained at the noise level in `config.yaml`, so the other levels are inputs it has not seen.

In [ ]:
if is_variational:

    set_seed(config["seed"])

    train_std = config["noise"]["train_val_std"]
    noise_levels = sorted({0.0, 0.1, train_std, 0.5})

    clean_images = next(iter(val_loader))[1].to(device)

    print(f"{'noise std':>10} {'input PSNR':>11} {'mean PSNR':>10} {'mean std':>10}")

    for noise_std in noise_levels:

        noisy = torch.clamp(
            clean_images + noise_std * torch.randn_like(clean_images),
            0.0,
            1.0,
        )

        mean_prediction, variance = predict_with_uncertainty(
            model,
            noisy,
            number_of_samples=30,
        )

        marker = "  <- training level" if noise_std == train_std else ""

        print(
            f"{noise_std:>10.2f} "
            f"{calculate_psnr(noisy, clean_images):>11.2f} "
            f"{calculate_psnr(mean_prediction, clean_images):>10.2f} "
            f"{variance.sqrt().mean().item():>10.5f}"
            f"{marker}"
        )

In [ ]:
save_history(
    history=history,
    run_dir=run_dir,
    test_loss=test_loss,
    test_psnr=test_psnr,
)